# Image Comparison Analysis - Bone Scan (Google Colab Demo)

This notebook is optimized for Google Colab. It compares two images and analyzes the similarity of specific color regions (red and blue).

## Step 1: Download Sample Images

Run this cell to automatically download sample images from GitHub:

In [ ]:
# Download sample images from GitHub
import urllib.request
import os

print("Downloading sample images...")

# Download origin.png
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/sojin25/bonescan/main/origin.png',
    'origin.png'
)
print("✓ Downloaded origin.png")

# Download filter.png
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/sojin25/bonescan/main/filter.png',
    'filter.png'
)
print("✓ Downloaded filter.png")

print("\nSample images are ready!")

## Step 2: Upload Your Own Images (Optional)

If you want to use your own images instead of the samples, uncomment and run the following cell:

In [ ]:
# Uncomment the following lines to upload your own images
# from google.colab import files

# print("Please upload your image files:")
# uploaded = files.upload()

# # Display uploaded files
# for filename in uploaded.keys():
#     print(f'Uploaded: {filename}')

# # Update file paths if you uploaded different files
# # img1_path = "your_origin_image.png"
# # img2_path = "your_filter_image.png"

## Step 3: Import Required Libraries

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.io import imread
from scipy.spatial.distance import directed_hausdorff

print("Libraries imported successfully!")

## Step 4: Define Analysis Functions

In [ ]:
def calculate_dice_coefficient(m1, m2):
    """Calculate Dice coefficient for two binary masks"""
    if m1.sum() + m2.sum() == 0:
        return 1.0  # Both empty = perfect match
    return 2.0 * np.logical_and(m1, m2).sum() / (m1.sum() + m2.sum())

def calculate_hausdorff_distance(m1, m2):
    """Calculate Hausdorff distance between two binary masks"""
    c1 = np.column_stack(np.where(m1))
    c2 = np.column_stack(np.where(m2))
    if c1.size == 0 or c2.size == 0:
        return np.nan  # Either empty = undefined distance
    return max(directed_hausdorff(c1, c2)[0],
               directed_hausdorff(c2, c1)[0])

def segment_color_regions(img, lower, upper):
    """Segment color regions based on HSV range"""
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    return cv2.inRange(hsv, lower, upper)

def create_overlay_image_with_white_background(img_ref, m1, m2, alpha=0.25):
    """Create overlay image showing matching and non-matching regions"""
    white_bg = np.ones_like(img_ref) * 255
    overlay = np.zeros_like(img_ref)
    overlay[np.logical_and(m1, m2)] = [0, 255, 0]   # green (matching)
    overlay[np.logical_xor(m1, m2)] = [255, 0, 0]   # red (non-matching)
    return cv2.addWeighted(white_bg, alpha, overlay, 1-alpha, 0)

def align_images_by_contours(img1, img2):
    """Align two images based on their largest contours"""
    g1 = cv2.cvtColor(img1, cv2.COLOR_RGB2GRAY)
    g2 = cv2.cvtColor(img2, cv2.COLOR_RGB2GRAY)
    _, t1 = cv2.threshold(g1, 0, 255, cv2.THRESH_OTSU)
    _, t2 = cv2.threshold(g2, 0, 255, cv2.THRESH_OTSU)
    cnt1, _ = cv2.findContours(t1, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cnt2, _ = cv2.findContours(t2, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    x1, y1, w1, h1 = cv2.boundingRect(max(cnt1, key=cv2.contourArea))
    x2, y2, w2, h2 = cv2.boundingRect(max(cnt2, key=cv2.contourArea))
    crop1 = img1[y1:y1+h1, x1:x1+w1]
    crop2 = cv2.resize(img2[y2:y2+h2, x2:x2+w2], (w1, h1))
    return crop1, crop2

print("Functions defined successfully!")

## Step 5: Set Image Paths

In [ ]:
# Image file paths
img1_path = "origin.png"
img2_path = "filter.png"

# Check if files exist
import os
if os.path.exists(img1_path) and os.path.exists(img2_path):
    print("✓ Image files found!")
    print(f"  - {img1_path}")
    print(f"  - {img2_path}")
else:
    print("⚠ Image files not found. Please run Step 1 to download sample images.")

## Step 6: Load and Process Images

In [ ]:
# Load images
img1 = imread(img1_path)
img2 = imread(img2_path)

# Display image information
print(f"Image 1 shape: {img1.shape}")
print(f"Image 2 shape: {img2.shape}")

# Align images
aligned1, aligned2 = align_images_by_contours(img1, img2)
print(f"\nAligned shape: {aligned1.shape}")

## Step 7: Display Original and Aligned Images

In [ ]:
# Display images
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].imshow(img1)
axes[0, 0].set_title("Original Image 1 (Origin)")
axes[0, 0].axis('off')

axes[0, 1].imshow(img2)
axes[0, 1].set_title("Original Image 2 (Filter)")
axes[0, 1].axis('off')

axes[1, 0].imshow(aligned1)
axes[1, 0].set_title("Aligned Image 1")
axes[1, 0].axis('off')

axes[1, 1].imshow(aligned2)
axes[1, 1].set_title("Aligned Image 2")
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

## Step 8: Define Color Ranges and Perform Segmentation

In [ ]:
# Define HSV color ranges
# Red color ranges (red hue is around 0° and 180°)
lower_red1 = np.array([0, 70, 50])
upper_red1 = np.array([10, 255, 255])
lower_red2 = np.array([170, 70, 50])
upper_red2 = np.array([180, 255, 255])

# Blue color range
lower_blue = np.array([100, 150, 0])
upper_blue = np.array([140, 255, 255])

# Segment color regions
red1 = segment_color_regions(aligned1, lower_red1, upper_red1) | \
       segment_color_regions(aligned1, lower_red2, upper_red2)
red2 = segment_color_regions(aligned2, lower_red1, upper_red1) | \
       segment_color_regions(aligned2, lower_red2, upper_red2)

blue1 = segment_color_regions(aligned1, lower_blue, upper_blue)
blue2 = segment_color_regions(aligned2, lower_blue, upper_blue)

# Display segmentation statistics
print("Segmentation Results:")
print(f"Red pixels in image 1: {np.sum(red1 > 0):,}")
print(f"Red pixels in image 2: {np.sum(red2 > 0):,}")
print(f"Blue pixels in image 1: {np.sum(blue1 > 0):,}")
print(f"Blue pixels in image 2: {np.sum(blue2 > 0):,}")

## Step 9: Calculate Similarity Metrics

In [ ]:
# Calculate similarity metrics
dice_r = calculate_dice_coefficient(red1 > 0, red2 > 0)
dice_b = calculate_dice_coefficient(blue1 > 0, blue2 > 0)
haus_r = calculate_hausdorff_distance(red1 > 0, red2 > 0)
haus_b = calculate_hausdorff_distance(blue1 > 0, blue2 > 0)

# Display results
print("=" * 60)
print("SIMILARITY METRICS")
print("=" * 60)
print("\nRed regions (Hot spots):")
print(f"  • Dice coefficient: {dice_r:.3f} (0=no overlap, 1=perfect match)")
if not np.isnan(haus_r):
    print(f"  • Hausdorff distance: {haus_r:.1f} pixels (lower=more similar)")
else:
    print(f"  • Hausdorff distance: N/A (empty region)")

print("\nBlue regions (Cold spots):")
print(f"  • Dice coefficient: {dice_b:.3f} (0=no overlap, 1=perfect match)")
if not np.isnan(haus_b):
    print(f"  • Hausdorff distance: {haus_b:.1f} pixels (lower=more similar)")
else:
    print(f"  • Hausdorff distance: N/A (empty region)")
print("=" * 60)

## Step 10: Visualize Results

In [ ]:
# Create visualization
fig, ax = plt.subplots(1, 2, figsize=(16, 8))

# Red regions overlay
ax[0].imshow(create_overlay_image_with_white_background(aligned1, red1 > 0, red2 > 0))
title_red = f"Red Regions (Hot Spots) Overlay\nDice: {dice_r:.3f}"
if not np.isnan(haus_r):
    title_red += f", Hausdorff: {haus_r:.1f}px"
ax[0].set_title(title_red, fontsize=14)
ax[0].axis('off')

# Blue regions overlay
ax[1].imshow(create_overlay_image_with_white_background(aligned1, blue1 > 0, blue2 > 0))
title_blue = f"Blue Regions (Cold Spots) Overlay\nDice: {dice_b:.3f}"
if not np.isnan(haus_b):
    title_blue += f", Hausdorff: {haus_b:.1f}px"
ax[1].set_title(title_blue, fontsize=14)
ax[1].axis('off')

# Add legend
fig.text(0.5, 0.02, '🟢 Green: Matching regions | 🔴 Red: Non-matching regions', 
         ha='center', fontsize=14, 
         bbox=dict(boxstyle='round,pad=0.5', facecolor='lightgray', alpha=0.8))

plt.tight_layout()
plt.show()

## Step 11: Save Results (Optional)

In [ ]:
# Uncomment the following lines to save and download results

# # Generate output images
# output_red = create_overlay_image_with_white_background(aligned1, red1 > 0, red2 > 0)
# output_blue = create_overlay_image_with_white_background(aligned1, blue1 > 0, blue2 > 0)

# # Save images
# cv2.imwrite('red_comparison.png', cv2.cvtColor(output_red, cv2.COLOR_RGB2BGR))
# cv2.imwrite('blue_comparison.png', cv2.cvtColor(output_blue, cv2.COLOR_RGB2BGR))
# print("✓ Results saved!")

# # Download files in Google Colab
# from google.colab import files
# files.download('red_comparison.png')
# files.download('blue_comparison.png')
# print("✓ Files downloaded!")

## Summary

This analysis compared two bone scan images and calculated:
- **Dice Coefficient**: Measures overlap between regions (0-1, higher is better)
- **Hausdorff Distance**: Measures shape similarity (lower is better)

The visualization shows:
- 🟢 **Green areas**: Regions that match between both images
- 🔴 **Red areas**: Regions that differ between images

To use your own images:
1. Run Step 2 to upload your files
2. Update the file paths in Step 5
3. Re-run all subsequent cells